# IAN 603 · Module 9
# Data Cleaning in PostgreSQL

Data cleaning is not the act of making every value look neat.

It is a decision process:

```text
profile
  ↓
standardize representation
  ↓
validate against rules
  ↓
flag uncertainty
  ↓
detect duplicates
  ↓
review / quarantine
  ↓
apply only defensible changes
  ↓
audit and validate
```

The core principle for this module is:

> **Do not erase uncertainty just to make the table look clean.**

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module9.html


## What you will learn

By the end of this notebook, you should be able to:

1. Profile data quality before changing anything.
2. Distinguish NULL, blank, sentinel, invalid, and suspicious values.
3. Normalize blank strings to NULL when appropriate.
4. Standardize names, emails, phones, and categories.
5. Explain the difference between standardization and correction.
6. Validate email, phone, product, and date fields with explicit rules.
7. Build reusable issue flags and issue reports.
8. Explain several common data-quality dimensions.
9. Create a simple illustrative quality score without treating it as universal truth.
10. Detect exact normalized duplicate groups.
11. Rank duplicate candidates using a documented survivor rule.
12. Distinguish duplicate detection from duplicate resolution.
13. Create quarantine / review views.
14. Create a clean preview layer without overwriting the raw source.
15. Use constraints to prevent known defects at data-entry time.
16. Test updates safely with transactions and `RETURNING`.
17. Use `IS DISTINCT FROM` to update only values that truly differ.
18. Build an auditable cleaning pipeline with issue codes and lineage.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** The helper is downloaded automatically when needed.

If you want your own copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request
from sqlalchemy import text

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Start with the Raw Source

This module intentionally includes imperfect values.

Examples include:

- NULL values,
- blank text,
- extra spaces,
- inconsistent capitalization,
- formatted and unformatted phone numbers,
- sentinel phone numbers,
- invalid emails,
- invalid prices,
- possible duplicate customers.

Before cleaning, we need to understand what is actually present.


## 4. Create or reset the Module 9 teaching database


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 9\n-- Data Cleaning in PostgreSQL\n-- ============================================================\n-- Companion file for s_module9.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- profiling, NULL/blank/sentinel values, standardization,\n-- validation, regex checks, quality flags, duplicate detection,\n-- survivor rules, quarantine/review, and safe updates.\n--\n-- Main principle: cleaning is evidence-based decision making.\n-- Standardize representation when justified; do not invent facts.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP VIEW IF EXISTS customers_clean_preview CASCADE;\nDROP VIEW IF EXISTS customer_quality_issues CASCADE;\nDROP TABLE IF EXISTS validated_contacts CASCADE;\nDROP TABLE IF EXISTS messy_customers CASCADE;\nDROP TABLE IF EXISTS orders CASCADE;\nDROP TABLE IF EXISTS products CASCADE;\nDROP TABLE IF EXISTS customers CASCADE;\n\nCREATE TABLE customers (\n    id integer PRIMARY KEY,\n    name text,\n    email text,\n    phone text,\n    address text,\n    signup_date date\n);\n\nCREATE TABLE products (\n    id integer PRIMARY KEY,\n    product_name text,\n    category text,\n    description text,\n    price numeric(10,2),\n    stock_quantity integer\n);\n\nCREATE TABLE orders (\n    id integer PRIMARY KEY,\n    customer_id integer REFERENCES customers(id),\n    order_date date NOT NULL,\n    total_amount numeric(10,2) CHECK (total_amount >= 0),\n    status text\n);\n\nCREATE TABLE messy_customers (\n    id integer PRIMARY KEY,\n    full_name text,\n    email text,\n    phone text,\n    signup_date_raw text\n);\n\nINSERT INTO customers VALUES\n(1,'John Doe','john.doe@email.com','(336) 555-1201','123 Main St',DATE '2023-01-15'),\n(2,'  JANE   SMITH  ',' JANE.SMITH@EMAIL.COM ','336.555.1202','456 Oak Ave',DATE '2023-02-20'),\n(3,'Bob Johnson',NULL,'3365551203',NULL,DATE '2023-03-10'),\n(4,'Alice Brown','alice.brown@test.com',NULL,'789 Pine Rd',DATE '2023-04-05'),\n(5,'CAROL WILSON','CAROL@TEST.COM','(336) 555-1205','101 Elm St',DATE '2023-05-12'),\n(6,'  David Lee  ','david.lee@example.com','336-555-1206','202 Maple Dr',DATE '2023-06-18'),\n(7,NULL,'empty.name@test.com','3365551207','303 Birch Ln',DATE '2023-07-22'),\n(8,'Duplicate User','duplicate@test.com','3365551208','404 Spruce St',DATE '2023-08-30'),\n(9,' Duplicate   User ',' DUPLICATE@TEST.COM ','(336) 555-1208',NULL,DATE '2023-09-01'),\n(10,'Test User','invalid-email','123','','2023-09-15'),\n(11,'Sentinel Phone','sentinel@example.com','0000000000','505 Cedar St',DATE '2023-10-01'),\n(12,'Blank Contact',' ',' ','',NULL);\n\nINSERT INTO products VALUES\n(1,'Laptop Computer','Electronics','High-performance laptop with 16GB RAM',999.99,10),\n(2,'Wireless Mouse','electronics','Ergonomic wireless mouse',29.99,25),\n(3,'Keyboard',NULL,'Mechanical keyboard',79.99,15),\n(4,'Monitor','Electronics','27 inch 4K monitor',-399.99,8),\n(5,'Headphones','AUDIO','Noise-cancelling headphones',199.99,NULL),\n(6,'','Electronics','Tablet device',299.99,5),\n(7,'Desk Lamp','Home Office','LED desk lamp',49.99,30),\n(8,'Notebook','stationery','Spiral notebook',NULL,100),\n(9,'Pen Set','Stationery','Set of 5 pens',12.99,50);\n\nINSERT INTO orders VALUES\n(101,1,DATE '2026-01-05',150.00,'shipped'),\n(102,2,DATE '2026-01-20',300.00,'pending'),\n(103,1,DATE '2026-02-10',200.00,'delivered'),\n(104,3,DATE '2026-03-04',450.00,'shipped'),\n(105,2,DATE '2026-03-18',125.00,'cancelled');\n\nINSERT INTO messy_customers VALUES\n(1,'  JOHN DOE  ','JOHN@EXAMPLE.COM','(555) 123-4567','2023-01-15'),\n(2,'jane smith','jane.smith@email.com','555.987.6543','01/20/2023'),\n(3,'bob johnson',NULL,'5551112233','2023-02-28'),\n(4,'Alice Brown','alice.brown@test.com',NULL,'03/15/2023'),\n(5,'  carol  wilson  ','CAROL@TEST.COM','(555) 444-5555','invalid_date'),\n(6,'David Lee','david.lee@example.com','555-999-8888','2023-04-01'),\n(7,'  ','empty.name@test.com','5550001111','2023-05-10'),\n(8,'Duplicate User','duplicate@test.com','5551112222','2023-06-15'),\n(9,' Duplicate   User ',' DUPLICATE@TEST.COM ','(555)111-2222','2023-06-15');"

execute_script(setup_sql)


## 5. Inspect the customer source rows


In [ ]:
display_query("""
SELECT
    id,
    name,
    email,
    phone,
    address,
    signup_date
FROM customers
ORDER BY id;
""", title="Raw customers")


## 6. Inspect the product source rows


In [ ]:
display_query("""
SELECT
    id,
    product_name,
    category,
    price,
    stock_quantity
FROM products
ORDER BY id;
""", title="Raw products")


# Part II · Profile Before You Change Anything

Profiling answers questions such as:

- How many rows are present?
- How many values are NULL?
- How many values are blank?
- Which categories occur?
- Which values repeat?
- Which values violate expected rules?

Do this before writing an `UPDATE`.


## 7. Profile missing and blank customer fields

Notice the difference between:

```sql
COUNT(email)
```

and:

```sql
COUNT(NULLIF(BTRIM(email), ''))
```

The second expression treats blank and whitespace-only text as unusable.


In [ ]:
display_query("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(name) AS nonnull_name,
    COUNT(NULLIF(BTRIM(name), '')) AS nonblank_name,

    COUNT(email) AS nonnull_email,
    COUNT(NULLIF(BTRIM(email), '')) AS nonblank_email,

    COUNT(phone) AS nonnull_phone,
    COUNT(NULLIF(BTRIM(phone), '')) AS nonblank_phone
FROM customers;
""", title="Missingness profile")


### Why this matters

A field can be:

| Raw state | Interpretation |
|---|---|
| NULL | missing / unknown |
| `''` | blank text stored |
| `'   '` | whitespace-only text |
| sentinel | placeholder such as 0000000000 |
| invalid value | present, but fails a rule |

Those states should not be collapsed automatically.


# Part III · Classify Missing, Blank, Sentinel, and Candidate Values

Before standardizing a phone number, classify what kind of value it is.


## 8. Classify phone states


In [ ]:
display_query("""
SELECT
    id,
    phone,
    CASE
        WHEN phone IS NULL
            THEN 'NULL'

        WHEN BTRIM(phone) = ''
            THEN 'blank'

        WHEN REGEXP_REPLACE(
                 phone,
                 '[^0-9]',
                 '',
                 'g'
             ) IN ('0000000000', '9999999999')
            THEN 'sentinel'

        ELSE 'candidate value'
    END AS phone_state
FROM customers
ORDER BY id;
""", title="Phone-state classification")


### Important

A sentinel value is not necessarily a real phone number.

Removing punctuation from:

```text
000-000-0000
```

does not make it valid.

Standardization and validation are separate steps.


# Part IV · Normalize Blank Text to NULL

When blank text has the same business meaning as “no usable value,” a useful pattern is:

```sql
NULLIF(BTRIM(column), '')
```

This:

1. trims the value,
2. turns an empty result into NULL.


## 9. Normalize blank addresses


In [ ]:
display_query("""
SELECT
    id,
    address AS raw_address,
    NULLIF(BTRIM(address), '') AS normalized_address
FROM customers
ORDER BY id;
""", title="Blank address normalization")


# Part V · COALESCE Is a Fallback, Not a Cleaning Rule

`COALESCE` returns the first non-NULL value.

It is useful only after we decide what should count as missing.

A contact fallback could be:

```text
usable email
   ↓
usable phone
   ↓
fallback label
```


## 10. Build a preferred contact value


In [ ]:
display_query("""
SELECT
    id,
    name,
    COALESCE(
        NULLIF(BTRIM(email), ''),
        NULLIF(BTRIM(phone), ''),
        'no contact method'
    ) AS preferred_contact
FROM customers
ORDER BY id;
""", title="Preferred contact fallback")


### Limitation

This fallback does not prove the email or phone is valid.

It only chooses the first nonblank representation.

Validation comes next.


# Part VI · Standardization vs Correction

These concepts are different.

### Standardization

Make equivalent representations consistent.

Examples:
- trim spaces,
- lowercase emails,
- keep only phone digits,
- map known category synonyms.

### Correction

Assert that one value is wrong and replace it with another.

Correction requires stronger evidence.

A cleaning pipeline should standardize confidently and flag uncertain corrections for review.


# Part VII · Standardize Names

For this teaching dataset, we will:

1. trim boundaries,
2. collapse repeated whitespace,
3. use title-style capitalization.

Keep the raw value beside the transformed value.


## 11. Standardize customer names


In [ ]:
display_query("""
SELECT
    id,
    name AS raw_name,
    INITCAP(
        REGEXP_REPLACE(
            BTRIM(name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    ) AS clean_name
FROM customers
ORDER BY id;
""", title="Name standardization")


# Part VIII · Standardize Emails

A conservative standardization is:

```text
trim boundaries
      ↓
convert blank to NULL
      ↓
lowercase
```

This changes representation, not validity.


## 12. Standardize email representation


In [ ]:
display_query("""
SELECT
    id,
    email AS raw_email,
    LOWER(
        NULLIF(
            BTRIM(email),
            ''
        )
    ) AS clean_email
FROM customers
ORDER BY id;
""", title="Email standardization")


# Part IX · Standardize Phones

Phone values may contain:

- parentheses,
- spaces,
- dashes,
- country codes,
- extensions,
- placeholders.

For this module's simplified U.S. example, we will extract digits and accept exactly 10 digits as a candidate standardized phone.


## 13. Convert phone text to digits


In [ ]:
display_query("""
SELECT
    id,
    phone AS raw_phone,
    NULLIF(
        REGEXP_REPLACE(
            COALESCE(phone, ''),
            '[^0-9]',
            '',
            'g'
        ),
        ''
    ) AS digits_only,
    CASE
        WHEN LENGTH(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            )
        ) = 10
        THEN REGEXP_REPLACE(
            phone,
            '[^0-9]',
            '',
            'g'
        )
        ELSE NULL
    END AS candidate_10_digit_phone
FROM customers
ORDER BY id;
""", title="Phone standardization")


### Why call it a candidate?

Ten digits satisfy our shape rule.

That does not prove:
- the number belongs to the customer,
- the number is active,
- the number is not a sentinel.

Validation and business review may still be needed.


# Part X · Standardize Known Categories

Categorical values often contain synonyms.

For example:

```text
electronics
electronic
tech
```

may all represent the same reporting category.

A documented mapping rule is better than silently applying `INITCAP` to everything.


## 14. Map product categories


In [ ]:
display_query("""
SELECT
    id,
    product_name,
    category AS raw_category,
    CASE
        WHEN NULLIF(BTRIM(category), '') IS NULL
            THEN NULL

        WHEN LOWER(BTRIM(category))
             IN ('electronics', 'electronic', 'tech')
            THEN 'Electronics'

        WHEN LOWER(BTRIM(category))
             IN ('audio', 'headphones')
            THEN 'Audio'

        WHEN LOWER(BTRIM(category))
             IN ('stationery', 'office supplies')
            THEN 'Stationery'

        WHEN LOWER(BTRIM(category))
             IN ('home office', 'home & garden')
            THEN 'Home Office'

        ELSE INITCAP(BTRIM(category))
    END AS standard_category
FROM products
ORDER BY id;
""", title="Category standardization")


### Keep the mapping visible

Category mapping is a business rule.

In a production system, you might maintain the mapping in a reference table instead of embedding it directly in a `CASE`.


# Part XI · Validation Is Rule-Based

Standardization asks:

> How should this value be represented?

Validation asks:

> Does this value satisfy an expected rule?

A value can be standardized and still invalid.


## 15. Validate email shape

This is a teaching-level shape rule, not proof that the mailbox exists.


In [ ]:
display_query("""
SELECT
    id,
    email,
    CASE
        WHEN NULLIF(BTRIM(email), '') IS NULL
            THEN 'missing'

        WHEN LOWER(BTRIM(email))
             ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
            THEN 'valid shape'

        ELSE 'review'
    END AS email_status
FROM customers
ORDER BY id;
""", title="Email validation")


## 16. Validate phone shape


In [ ]:
display_query("""
SELECT
    id,
    phone,
    REGEXP_REPLACE(
        COALESCE(phone, ''),
        '[^0-9]',
        '',
        'g'
    ) AS digits,
    CASE
        WHEN NULLIF(BTRIM(phone), '') IS NULL
            THEN 'missing'

        WHEN LENGTH(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            )
        ) <> 10
            THEN 'review'

        WHEN REGEXP_REPLACE(
                 phone,
                 '[^0-9]',
                 '',
                 'g'
             ) IN ('0000000000', '9999999999')
            THEN 'sentinel - review'

        ELSE 'valid shape'
    END AS phone_status
FROM customers
ORDER BY id;
""", title="Phone validation")


## 17. Validate product fields


In [ ]:
display_query("""
SELECT
    id,
    product_name,
    category,
    price,
    stock_quantity,
    CASE
        WHEN NULLIF(BTRIM(product_name), '') IS NULL
            THEN 'missing product name'

        WHEN price IS NULL
            THEN 'missing price'

        WHEN price <= 0
            THEN 'invalid price'

        WHEN stock_quantity IS NOT NULL
             AND stock_quantity < 0
            THEN 'negative stock'

        ELSE 'passes selected rules'
    END AS validation_status
FROM products
ORDER BY id;
""", title="Product validation")


# Part XII · Turn Validation into an Issue Report

A row may contain more than one problem.

Instead of forcing one overall label, an issue table can create **one row per issue**.

That structure is easier to:
- count,
- filter,
- assign,
- trend over time.


## 18. Create an issue report


In [ ]:
display_query("""
WITH issues AS (
    SELECT
        id,
        'customer' AS entity,
        'invalid_email' AS issue
    FROM customers
    WHERE NULLIF(BTRIM(email), '') IS NOT NULL
      AND LOWER(BTRIM(email))
          !~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'

    UNION ALL

    SELECT
        id,
        'customer',
        'invalid_phone'
    FROM customers
    WHERE NULLIF(BTRIM(phone), '') IS NOT NULL
      AND (
          LENGTH(
              REGEXP_REPLACE(
                  phone,
                  '[^0-9]',
                  '',
                  'g'
              )
          ) <> 10
          OR REGEXP_REPLACE(
                 phone,
                 '[^0-9]',
                 '',
                 'g'
             ) IN ('0000000000', '9999999999')
      )

    UNION ALL

    SELECT
        id,
        'product',
        'invalid_price'
    FROM products
    WHERE price IS NULL
       OR price <= 0
)
SELECT
    entity,
    issue,
    COUNT(*) AS issue_count
FROM issues
GROUP BY entity, issue
ORDER BY issue_count DESC, entity, issue;
""", title="Quality issue summary")


# Part XIII · Data-Quality Dimensions

Common quality dimensions include:

| Dimension | Example question |
|---|---|
| completeness | Is a required value present? |
| validity | Does it satisfy the rule? |
| consistency | Is the representation standardized? |
| uniqueness | Is this entity duplicated? |
| timeliness | Is the value current enough? |
| accuracy | Does it reflect reality? |

Not every dimension can be measured from one database table alone.

For example, a syntactically valid email may still be inaccurate.


# Part XIV · Illustrative Quality Scores

A score can summarize several checks, but a score is a **model**, not truth.

For teaching purposes, we will give 25 points each for:

- usable name,
- valid-shape email,
- valid-shape phone,
- plausible signup date.

This weighting is illustrative only.


## 19. Calculate an illustrative quality score


In [ ]:
display_query("""
SELECT
    id,
    name,
    (
        CASE
            WHEN NULLIF(BTRIM(name), '') IS NOT NULL
                THEN 25 ELSE 0
        END
        +
        CASE
            WHEN LOWER(COALESCE(BTRIM(email), ''))
                 ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
                THEN 25 ELSE 0
        END
        +
        CASE
            WHEN LENGTH(
                REGEXP_REPLACE(
                    COALESCE(phone, ''),
                    '[^0-9]',
                    '',
                    'g'
                )
            ) = 10
            AND REGEXP_REPLACE(
                    COALESCE(phone, ''),
                    '[^0-9]',
                    '',
                    'g'
                ) NOT IN ('0000000000', '9999999999')
                THEN 25 ELSE 0
        END
        +
        CASE
            WHEN signup_date IS NOT NULL
             AND signup_date <= DATE '2026-08-12'
                THEN 25 ELSE 0
        END
    ) AS illustrative_quality_score
FROM customers
ORDER BY illustrative_quality_score, id;
""", title="Illustrative quality score")


## 20. Summarize quality bands

The band labels are also business choices.


In [ ]:
display_query("""
WITH scored AS (
    SELECT
        id,
        (
            CASE
                WHEN NULLIF(BTRIM(name), '') IS NOT NULL
                    THEN 25 ELSE 0
            END
            +
            CASE
                WHEN LOWER(COALESCE(BTRIM(email), ''))
                     ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
                    THEN 25 ELSE 0
            END
            +
            CASE
                WHEN LENGTH(
                    REGEXP_REPLACE(
                        COALESCE(phone, ''),
                        '[^0-9]',
                        '',
                        'g'
                    )
                ) = 10
                    THEN 25 ELSE 0
            END
            +
            CASE
                WHEN signup_date IS NOT NULL
                 AND signup_date <= DATE '2026-08-12'
                    THEN 25 ELSE 0
            END
        ) AS score
    FROM customers
)
SELECT
    CASE
        WHEN score = 100 THEN 'Excellent'
        WHEN score >= 75 THEN 'Good'
        WHEN score >= 50 THEN 'Fair'
        ELSE 'Poor'
    END AS quality_band,
    COUNT(*) AS customer_count
FROM scored
GROUP BY 1
ORDER BY MIN(score) DESC;
""", title="Quality band summary")


# Part XV · Duplicate Detection Starts with a Rule

A duplicate is not simply “two similar-looking rows.”

You need a duplicate key or matching rule.

For this example:

```text
normalized email
+
normalized name
```

defines an exact normalized duplicate group.

That rule may be appropriate for teaching, but a real identity-resolution system may need stronger evidence.


## 21. Detect exact normalized duplicate groups


In [ ]:
display_query("""
SELECT
    LOWER(BTRIM(email)) AS email_key,
    INITCAP(
        REGEXP_REPLACE(
            BTRIM(name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    ) AS name_key,
    COUNT(*) AS row_count,
    ARRAY_AGG(id ORDER BY id) AS source_ids
FROM customers
WHERE NULLIF(BTRIM(email), '') IS NOT NULL
GROUP BY
    LOWER(BTRIM(email)),
    INITCAP(
        REGEXP_REPLACE(
            BTRIM(name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    )
HAVING COUNT(*) > 1;
""", title="Normalized duplicate groups")


# Part XVI · Detection Is Not Resolution

Finding duplicate candidates does **not** tell us which record should survive.

Resolution needs a documented rule.

Possible evidence:
- completeness,
- recency,
- trusted source,
- verified contact,
- business system priority.

In this teaching example, we will rank candidates by completeness, then recency.


## 22. Score completeness before choosing a survivor


In [ ]:
display_query("""
SELECT
    id,
    name,
    email,
    phone,
    address,
    (
        CASE WHEN NULLIF(BTRIM(name), '') IS NOT NULL THEN 1 ELSE 0 END
        +
        CASE WHEN NULLIF(BTRIM(email), '') IS NOT NULL THEN 1 ELSE 0 END
        +
        CASE WHEN NULLIF(BTRIM(phone), '') IS NOT NULL THEN 1 ELSE 0 END
        +
        CASE WHEN NULLIF(BTRIM(address), '') IS NOT NULL THEN 1 ELSE 0 END
    ) AS completeness_score,
    signup_date
FROM customers
WHERE LOWER(BTRIM(email)) = 'duplicate@test.com'
ORDER BY completeness_score DESC, signup_date DESC NULLS LAST, id;
""", title="Duplicate candidates with completeness score")


## 23. Rank duplicate survivors explicitly


In [ ]:
display_query("""
WITH scored AS (
    SELECT
        c.*,
        (
            CASE WHEN NULLIF(BTRIM(name), '') IS NOT NULL THEN 1 ELSE 0 END
            +
            CASE WHEN NULLIF(BTRIM(email), '') IS NOT NULL THEN 1 ELSE 0 END
            +
            CASE WHEN NULLIF(BTRIM(phone), '') IS NOT NULL THEN 1 ELSE 0 END
            +
            CASE WHEN NULLIF(BTRIM(address), '') IS NOT NULL THEN 1 ELSE 0 END
        ) AS completeness_score
    FROM customers AS c
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                LOWER(BTRIM(email)),
                INITCAP(
                    REGEXP_REPLACE(
                        BTRIM(name),
                        '[[:space:]]+',
                        ' ',
                        'g'
                    )
                )
            ORDER BY
                completeness_score DESC,
                signup_date DESC NULLS LAST,
                id
        ) AS survivor_rank
    FROM scored
)
SELECT
    id,
    name,
    email,
    completeness_score,
    signup_date,
    survivor_rank
FROM ranked
WHERE LOWER(BTRIM(email)) = 'duplicate@test.com'
ORDER BY survivor_rank;
""", title="Duplicate survivor ranking")


### Important wording

`survivor_rank = 1` means:

> this row won **our stated ranking rule**.

It does not prove the row is the true real-world identity.

That may still require review.


# Part XVII · Possible Duplicates Need Review

Exact normalized matches are only one type of duplicate evidence.

Two rows with the same normalized phone but different names or emails may be:

- the same person,
- family members sharing a phone,
- a recycled number,
- data-entry error.

Flag them. Do not merge automatically without a justified rule.


## 24. Find possible duplicate pairs by normalized phone


In [ ]:
display_query("""
SELECT
    a.id AS left_id,
    b.id AS right_id,
    a.name AS left_name,
    b.name AS right_name,
    a.email AS left_email,
    b.email AS right_email,
    REGEXP_REPLACE(
        COALESCE(a.phone, ''),
        '[^0-9]',
        '',
        'g'
    ) AS shared_phone_key
FROM customers AS a
JOIN customers AS b
    ON a.id < b.id
WHERE REGEXP_REPLACE(
          COALESCE(a.phone, ''),
          '[^0-9]',
          '',
          'g'
      )
      =
      REGEXP_REPLACE(
          COALESCE(b.phone, ''),
          '[^0-9]',
          '',
          'g'
      )
  AND NULLIF(
          REGEXP_REPLACE(
              COALESCE(a.phone, ''),
              '[^0-9]',
              '',
              'g'
          ),
          ''
      ) IS NOT NULL
ORDER BY a.id, b.id;
""", title="Possible duplicates by phone")


# Part XVIII · Quarantine Instead of Silent Repair

When a row clearly needs review, create a review layer.

A quarantine view lets us:

- keep the source untouched,
- make issues visible,
- route problematic rows for review,
- reuse the same quality rules.


## 25. Create a reusable customer quality issue view


In [ ]:
execute_script("""
CREATE OR REPLACE VIEW customer_quality_issues AS
SELECT
    id,
    name,
    email,
    phone,

    CASE
        WHEN NULLIF(BTRIM(name), '') IS NULL
            THEN true ELSE false
    END AS missing_name,

    CASE
        WHEN NULLIF(BTRIM(email), '') IS NOT NULL
         AND LOWER(BTRIM(email))
             !~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
            THEN true ELSE false
    END AS invalid_email,

    CASE
        WHEN NULLIF(BTRIM(phone), '') IS NOT NULL
         AND (
             LENGTH(
                 REGEXP_REPLACE(
                     phone,
                     '[^0-9]',
                     '',
                     'g'
                 )
             ) <> 10
             OR REGEXP_REPLACE(
                    phone,
                    '[^0-9]',
                    '',
                    'g'
                ) IN ('0000000000', '9999999999')
         )
            THEN true ELSE false
    END AS invalid_phone

FROM customers;
""")

display_query("""
SELECT *
FROM customer_quality_issues
WHERE missing_name
   OR invalid_email
   OR invalid_phone
ORDER BY id;
""", title="Customer rows needing review")


# Part XIX · Create a Clean Preview Layer

A clean preview view gives analysts standardized columns without overwriting source evidence.

This is often safer than immediately updating the raw table.


## 26. Create customers_clean_preview


In [ ]:
execute_script("""
CREATE OR REPLACE VIEW customers_clean_preview AS
SELECT
    id,

    INITCAP(
        REGEXP_REPLACE(
            BTRIM(name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    ) AS clean_name,

    LOWER(
        NULLIF(
            BTRIM(email),
            ''
        )
    ) AS clean_email,

    CASE
        WHEN LENGTH(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            )
        ) = 10
         AND REGEXP_REPLACE(
                 COALESCE(phone, ''),
                 '[^0-9]',
                 '',
                 'g'
             ) NOT IN ('0000000000', '9999999999')
        THEN REGEXP_REPLACE(
            phone,
            '[^0-9]',
            '',
            'g'
        )
        ELSE NULL
    END AS clean_phone,

    NULLIF(
        BTRIM(address),
        ''
    ) AS clean_address,

    signup_date

FROM customers;
""")

display_query("""
SELECT *
FROM customers_clean_preview
ORDER BY id;
""", title="Clean preview layer")


### Source vs analytical layer

```text
customers
   raw evidence
       ↓
customers_clean_preview
   standardized analytical representation
       ↓
reports / analysis
```

This preserves traceability.


# Part XX · Constraints Prevent Known Defects

Cleaning fixes or flags existing problems.

Constraints help prevent known problems from entering a controlled table in the first place.

Examples:
- `NOT NULL`
- `CHECK`
- `UNIQUE`
- foreign keys.


## 27. Create a validated contact table


In [ ]:
execute_script("""
DROP TABLE IF EXISTS validated_contacts;

CREATE TABLE validated_contacts (
    contact_id integer PRIMARY KEY,
    email text NOT NULL,
    phone text,

    CONSTRAINT email_shape_chk
        CHECK (
            email
            ~ '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
        ),

    CONSTRAINT phone_digits_chk
        CHECK (
            phone IS NULL
            OR phone ~ '^[0-9]{10}$'
        ),

    CONSTRAINT email_unique
        UNIQUE (email)
);

INSERT INTO validated_contacts
VALUES (
    1,
    'student@example.com',
    '3365551212'
);
""")

display_query("""
SELECT *
FROM validated_contacts;
""", title="Validated contact example")


## 28. Demonstrate a constraint safely

This invalid insert is expected to fail.


In [ ]:
try:
    with db.engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO validated_contacts
            VALUES (
                2,
                'not-an-email',
                '123'
            );
        """))
except Exception as e:
    print("✅ PostgreSQL rejected the invalid row.")
    print(str(e).split("\n")[0])


### Important timing

Add constraints after you understand the real data and the intended business rule.

A poorly designed constraint can reject legitimate values just as easily as it rejects bad ones.


# Part XXI · Safe Updates

Sometimes a standardization is strong enough to apply to the source.

Even then, use a safe workflow:

```text
preview
  ↓
BEGIN
  ↓
UPDATE ... RETURNING
  ↓
inspect
  ↓
COMMIT or ROLLBACK
```

For the notebook, we will roll the demonstration back.


## 29. Preview which emails would change


In [ ]:
display_query("""
SELECT
    id,
    email AS raw_email,
    LOWER(BTRIM(email)) AS proposed_email
FROM customers
WHERE email IS NOT NULL
  AND email IS DISTINCT FROM LOWER(BTRIM(email))
ORDER BY id;
""", title="Email changes to review")


## 30. Test the update inside a transaction

`IS DISTINCT FROM` is useful here because it compares values NULL-safely.


In [ ]:
with db.engine.connect() as conn:
    tx = conn.begin()

    try:
        result = conn.execute(text("""
            UPDATE customers
            SET email = LOWER(BTRIM(email))
            WHERE email IS NOT NULL
              AND email IS DISTINCT FROM LOWER(BTRIM(email))
            RETURNING id, email;
        """))

        rows = result.mappings().all()

        print("Rows that would be updated:")
        for row in rows:
            print(dict(row))

    finally:
        tx.rollback()

print("↩ Changes rolled back. Source customers were not permanently modified.")


# Part XXII · Do Not Overwrite Uncertainty

Some transformations are deterministic:

```text
'  Alice  ' → 'Alice'
```

Some are uncertain:

```text
'336555121' → what is the missing digit?
```

Do not invent missing information just to make a value pass validation.

Better choices include:
- keep the raw value,
- set a clean candidate to NULL,
- add an issue flag,
- send the row to review.


# Part XXIII · Issue Codes and Lineage

A mature cleaning pipeline should explain **why** a row was changed or flagged.

Useful fields can include:

- source row ID,
- raw value,
- cleaned value,
- issue code,
- cleaning rule,
- review status,
- processing timestamp.

For this notebook, source IDs provide the basic lineage back to raw records.


# Part XXIV · Integrated Cleaning Pipeline

Now combine the module into a staged pipeline.

The source table `messy_customers` contains:
- messy names,
- messy emails,
- messy phones,
- raw date text.

We will create three stages:

```text
normalized
    ↓
typed
    ↓
ranked
    ↓
final analytical output
```


## 31. Stage 1 — normalize representations

The first stage:
- normalizes whitespace,
- lowercases email,
- extracts phone digits,
- keeps date as raw text for now.


In [ ]:
display_query("""
WITH normalized AS (
    SELECT
        id,
        NULLIF(
            REGEXP_REPLACE(
                BTRIM(full_name),
                '[[:space:]]+',
                ' ',
                'g'
            ),
            ''
        ) AS name_norm,
        LOWER(
            NULLIF(
                BTRIM(email),
                ''
            )
        ) AS email_norm,
        NULLIF(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            ),
            ''
        ) AS phone_digits,
        signup_date_raw
    FROM messy_customers
)
SELECT *
FROM normalized
ORDER BY id;
""", title="Stage 1: normalized")


## 32. Stage 2 — type only recognized date formats

Unknown formats remain NULL for review instead of being guessed.


In [ ]:
display_query("""
WITH normalized AS (
    SELECT
        id,
        NULLIF(
            REGEXP_REPLACE(
                BTRIM(full_name),
                '[[:space:]]+',
                ' ',
                'g'
            ),
            ''
        ) AS name_norm,
        LOWER(NULLIF(BTRIM(email), '')) AS email_norm,
        NULLIF(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            ),
            ''
        ) AS phone_digits,
        signup_date_raw
    FROM messy_customers
),
typed AS (
    SELECT
        *,
        CASE
            WHEN signup_date_raw
                 ~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'
                THEN signup_date_raw::date

            WHEN signup_date_raw
                 ~ '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
                THEN TO_DATE(
                    signup_date_raw,
                    'MM/DD/YYYY'
                )

            ELSE NULL
        END AS signup_date_clean
    FROM normalized
)
SELECT *
FROM typed
ORDER BY id;
""", title="Stage 2: typed")


## 33. Stage 3 — rank duplicate candidates

The duplicate key is:

```text
normalized email
+
normalized name
```

The survivor rule favors more complete rows.


In [ ]:
display_query("""
WITH normalized AS (
    SELECT
        id,
        NULLIF(
            REGEXP_REPLACE(
                BTRIM(full_name),
                '[[:space:]]+',
                ' ',
                'g'
            ),
            ''
        ) AS name_norm,
        LOWER(NULLIF(BTRIM(email), '')) AS email_norm,
        NULLIF(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            ),
            ''
        ) AS phone_digits,
        signup_date_raw
    FROM messy_customers
),
typed AS (
    SELECT
        *,
        CASE
            WHEN signup_date_raw
                 ~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'
                THEN signup_date_raw::date
            WHEN signup_date_raw
                 ~ '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
                THEN TO_DATE(
                    signup_date_raw,
                    'MM/DD/YYYY'
                )
            ELSE NULL
        END AS signup_date_clean
    FROM normalized
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                email_norm,
                INITCAP(name_norm)
            ORDER BY
                (
                    CASE WHEN name_norm IS NOT NULL THEN 1 ELSE 0 END
                    +
                    CASE WHEN email_norm IS NOT NULL THEN 1 ELSE 0 END
                    +
                    CASE WHEN LENGTH(phone_digits) = 10 THEN 1 ELSE 0 END
                    +
                    CASE WHEN signup_date_clean IS NOT NULL THEN 1 ELSE 0 END
                ) DESC,
                id
        ) AS duplicate_rank
    FROM typed
)
SELECT *
FROM ranked
ORDER BY duplicate_rank, id;
""", title="Stage 3: duplicate ranking")


## 34. Final analytical output

The final result keeps:
- cleaned values,
- duplicate rank,
- an explicit review/candidate status.

It does not overwrite the raw source.


In [ ]:
display_query("""
WITH normalized AS (
    SELECT
        id,
        NULLIF(
            REGEXP_REPLACE(
                BTRIM(full_name),
                '[[:space:]]+',
                ' ',
                'g'
            ),
            ''
        ) AS name_norm,
        LOWER(NULLIF(BTRIM(email), '')) AS email_norm,
        NULLIF(
            REGEXP_REPLACE(
                COALESCE(phone, ''),
                '[^0-9]',
                '',
                'g'
            ),
            ''
        ) AS phone_digits,
        signup_date_raw
    FROM messy_customers
),
typed AS (
    SELECT
        *,
        CASE
            WHEN signup_date_raw
                 ~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'
                THEN signup_date_raw::date
            WHEN signup_date_raw
                 ~ '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
                THEN TO_DATE(
                    signup_date_raw,
                    'MM/DD/YYYY'
                )
            ELSE NULL
        END AS signup_date_clean
    FROM normalized
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                email_norm,
                INITCAP(name_norm)
            ORDER BY
                (
                    CASE WHEN name_norm IS NOT NULL THEN 1 ELSE 0 END
                    +
                    CASE WHEN email_norm IS NOT NULL THEN 1 ELSE 0 END
                    +
                    CASE WHEN LENGTH(phone_digits) = 10 THEN 1 ELSE 0 END
                    +
                    CASE WHEN signup_date_clean IS NOT NULL THEN 1 ELSE 0 END
                ) DESC,
                id
        ) AS duplicate_rank
    FROM typed
)
SELECT
    id,
    INITCAP(name_norm) AS clean_name,
    email_norm AS clean_email,
    CASE
        WHEN LENGTH(phone_digits) = 10
         AND phone_digits NOT IN ('0000000000', '9999999999')
            THEN phone_digits
        ELSE NULL
    END AS clean_phone,
    signup_date_clean,
    duplicate_rank,
    CASE
        WHEN name_norm IS NULL
          OR email_norm IS NULL
          OR signup_date_clean IS NULL
          OR (
              phone_digits IS NOT NULL
              AND (
                  LENGTH(phone_digits) <> 10
                  OR phone_digits IN ('0000000000', '9999999999')
              )
          )
            THEN 'review'
        ELSE 'candidate clean row'
    END AS row_status
FROM ranked
ORDER BY duplicate_rank, id;
""", title="Final cleaned analytical output")


# Part XXV · Validate Before and After

Cleaning is incomplete until we validate the result.


## 35. Profile unusable contact fields


In [ ]:
display_query("""
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (
        WHERE name IS NULL
    ) AS null_names,
    COUNT(*) FILTER (
        WHERE NULLIF(BTRIM(name), '') IS NULL
    ) AS unusable_names,
    COUNT(*) FILTER (
        WHERE NULLIF(BTRIM(email), '') IS NULL
    ) AS unusable_emails,
    COUNT(*) FILTER (
        WHERE NULLIF(BTRIM(phone), '') IS NULL
    ) AS unusable_phones
FROM customers;
""", title="Contact completeness profile")


## 36. Validate duplicate normalized emails


In [ ]:
display_query("""
SELECT
    LOWER(BTRIM(email)) AS normalized_email,
    COUNT(*) AS row_count,
    ARRAY_AGG(id ORDER BY id) AS source_ids
FROM customers
WHERE NULLIF(BTRIM(email), '') IS NOT NULL
GROUP BY LOWER(BTRIM(email))
HAVING COUNT(*) > 1
ORDER BY normalized_email;
""", title="Duplicate normalized emails")


## 37. Keep invalid rows visible for review


In [ ]:
display_query("""
SELECT
    id,
    name,
    email,
    phone
FROM customers
WHERE NULLIF(BTRIM(name), '') IS NULL
   OR (
       NULLIF(BTRIM(email), '') IS NOT NULL
       AND LOWER(BTRIM(email))
           !~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
   )
   OR (
       NULLIF(BTRIM(phone), '') IS NOT NULL
       AND (
           LENGTH(
               REGEXP_REPLACE(
                   phone,
                   '[^0-9]',
                   '',
                   'g'
               )
           ) <> 10
           OR REGEXP_REPLACE(
                  phone,
                  '[^0-9]',
                  '',
                  'g'
              ) IN ('0000000000', '9999999999')
       )
   )
ORDER BY id;
""", title="Rows requiring review")


### Validation checklist

After cleaning, ask:

1. How many rows were in the source?
2. How many candidate clean rows remain?
3. How many rows need review?
4. Did normalization create new duplicate keys?
5. Did any valid-looking source value become NULL?
6. Are sentinel values still being mistaken for valid values?
7. Can every cleaned row be traced to a source ID?
8. Were uncertain values flagged instead of invented?


# Part XXVI · Guided Practice

For every task, decide whether it is:

- profiling,
- standardization,
- validation,
- duplicate detection,
- duplicate resolution,
- quarantine,
- source update.

That decision should come before the SQL.


## Practice 1 · Profile customer contact quality

Return:

- total customers,
- unusable names,
- unusable emails,
- unusable phones.

Treat blank text as unusable.


In [ ]:
practice_1 = """
SELECT
    -- metrics
FROM customers;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · Standardize customer contact fields

Return one row per customer with:

- raw name
- clean name
- raw email
- clean email
- raw phone
- digits-only phone

Do not overwrite the source table.


In [ ]:
practice_2 = """
SELECT
    id,
    -- raw and cleaned fields
FROM customers
ORDER BY id;
"""


## Practice 3 · Product validation

Flag:

- missing product name,
- missing price,
- non-positive price,
- negative stock.

Do not replace uncertain values.


In [ ]:
practice_3 = """
SELECT
    id,
    product_name,
    price,
    stock_quantity,
    CASE
        -- validation rules
    END AS validation_status
FROM products
ORDER BY id;
"""


## Practice 4 · Duplicate detection

Find normalized duplicate groups using:

- normalized email,
- normalized name.

Return:
- duplicate key,
- row count,
- source IDs.


In [ ]:
practice_4 = """
SELECT
    -- normalized duplicate key,
    COUNT(*) AS row_count,
    ARRAY_AGG(id ORDER BY id) AS source_ids
FROM customers
WHERE
    -- usable duplicate key
GROUP BY
    -- normalized key
HAVING COUNT(*) > 1;
"""


## Practice 5 · Quarantine view

Create a view that flags:

- missing name,
- invalid email shape,
- invalid phone shape or sentinel phone.

Then query only rows with at least one issue.


In [ ]:
practice_5 = """
CREATE OR REPLACE VIEW practice_customer_issues AS
SELECT
    -- fields and flags
FROM customers;
"""

# execute_script(practice_5)


# Part XXVII · Final Challenge

Return **one row per customer** with:

- source customer ID,
- raw name,
- clean name,
- raw email,
- clean email,
- raw phone,
- clean phone candidate,
- email status,
- phone status,
- completeness score,
- duplicate rank,
- overall row status.

### Requirements

1. Do not update the source table.
2. Preserve raw values.
3. Treat blank text as missing where appropriate.
4. Do not treat sentinel phones as valid.
5. Rank duplicate candidates with a documented rule.
6. Use `review` when a value is uncertain or invalid.
7. Keep source IDs for lineage.


In [ ]:
final_query = """
WITH normalized AS (
    SELECT
        -- raw and normalized fields
    FROM customers
),
scored AS (
    SELECT
        *,
        -- completeness score
    FROM normalized
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                -- duplicate key
            ORDER BY
                -- survivor rule
        ) AS duplicate_rank
    FROM scored
)
SELECT
    -- final analytical output
FROM ranked
ORDER BY id;
"""

# display_query(final_query, title="Final challenge")


# Quick Concept Check

### Is NULL the same as blank text?
<details><summary>Show answer</summary>No. They are different stored states even if a cleaning rule may later normalize blank text to NULL.</details>

### Is standardization the same as correction?
<details><summary>Show answer</summary>No. Standardization makes equivalent representations consistent; correction asserts that one value should be replaced by another.</details>

### Does a 10-digit phone shape prove the phone is real?
<details><summary>Show answer</summary>No. It only satisfies the selected structural rule.</details>

### Does duplicate detection prove two rows represent the same real person?
<details><summary>Show answer</summary>No. It identifies candidates under a matching rule. Resolution may require more evidence.</details>

### Why keep raw and cleaned values together?
<details><summary>Show answer</summary>For auditability, traceability, and review of the transformation rule.</details>

### Why use a quarantine or issue view?
<details><summary>Show answer</summary>To keep questionable rows visible for review without silently overwriting the raw source.</details>

### What does IS DISTINCT FROM help with?
<details><summary>Show answer</summary>NULL-safe comparison, useful for detecting values that truly differ before an update.</details>

### Why should a cleaning pipeline have source IDs?
<details><summary>Show answer</summary>So every cleaned or flagged row can be traced back to the original record.</details>


# Module 9 Summary

A defensible cleaning workflow is:

1. Profile the raw data.
2. Distinguish NULL, blank, sentinel, invalid, and suspicious values.
3. Standardize representation where the rule is clear.
4. Validate with explicit business rules.
5. Keep uncertain values visible.
6. Create issue flags and review layers.
7. Define duplicate keys explicitly.
8. Separate duplicate detection from duplicate resolution.
9. Rank survivor candidates with documented evidence.
10. Preserve raw values and source IDs.
11. Use views for clean analytical layers.
12. Add constraints to prevent known defects when appropriate.
13. Test source updates inside transactions.
14. Validate before and after cleaning.

The goal is not:

> “make every field look complete.”

The goal is:

> **create a defensible analytical representation while preserving evidence and uncertainty.**

## Next
**Module 10: Reshaping Data in PostgreSQL**
